# GeoNUSAF — Dataset Characterisation (paper Section III: *Study Area and Dataset*)

**Kathmandu Valley satellite land-use segmentation — 6 classes + ignore**

Structural twin of the baseline notebooks: same 22-section layout, same `@dataclass CFG` pattern,
same asserts, same markdown-then-code alternation. The difference is that **nothing here trains,
loads a checkpoint, or touches a GPU**. Every cell is read-only with respect to the dataset. The
single job of this notebook is to produce every number, table and figure that Section III of the
paper needs, and to dump them to disk as JSON / CSV / LaTeX so the section can be written from
files rather than from screenshots.

## What it computes
| Block | Sections | What it gives the paper |
|---|---|---|
| Inventory | 3 | file counts, on-disk resolution, bit depth, dataset size, export-sequence gaps |
| Geometry | 5 | native and working GSD, tile footprint, annotated area, exported extent |
| Label integrity | 4, 7 | exhaustive palette audit over **every** mask, unmapped-pixel share |
| Resampling | 8 | how much the 2345 -> 512 downsample distorts each class (Road/River especially) |
| Class statistics | 9, 10, 11 | pixel shares, imbalance ratio, per-tile presence, co-occurrence |
| Spectral | 12 | per-class RGB statistics + Jeffries-Matusita separability (explains the confusions) |
| Structure | 13, 14 | patch density, edge density, diversity, Road/River width and length in metres |
| Imagery | 15 | dataset RGB mean/std, exposure, sharpness |
| Integrity | 16, 17 | flip-variant verification, near-duplicate detection |
| Protocol | 18, 19 | per-fold split composition, export-order autocorrelation |
| Outputs | 20, 21 | 300-dpi figures, `dataset_stats.json`, CSVs, ready-to-paste LaTeX tables |

## What it deliberately does not do
No model, no loss, no optimizer, no W&B, no Hugging Face. It does not write to `DATA_ROOT`.
The dataset is frozen; this notebook only measures it.

## Cost
One heavy pass decodes the **408 original tiles** at full resolution (images and masks, ~8 GB of
PNG) and caches the derived 512 arrays to `/kaggle/working/dsstats_cache.npz`. Budget **12-18
minutes** on Kaggle's 4 vCPUs the first time; every later run of the notebook reloads the npz in
~15 s. The 816 flip variants are exact mirrors of the originals (verified in section 16), so all
statistics are reported on the 408 independent tiles — reporting over 1224 would triple every
count while leaving every percentage unchanged, and would overstate the dataset's independence.

## Prerequisites
- Kaggle dataset `sugam24/satellite-datasets` attached.
- Internet **not** required. GPU **not** required (set the accelerator to None to save quota).
- `scikit-image`, `pandas`, `matplotlib`, `opencv` are all preinstalled on Kaggle.

## 1. Environment

Nothing is installed. `scipy`, `skimage` and `pandas` ship with the Kaggle image; their versions
are printed rather than pinned so the exact values can go in the reproducibility paragraph.
`cv2.setNumThreads(0)` matters here for the same reason it matters in the training notebooks —
OpenCV's internal pool fights the `ProcessPoolExecutor` in section 6 on 4 vCPUs.

In [ ]:
import os, re, gc, json, time, math, random, warnings, itertools
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import cv2
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import skimage
from skimage.morphology import skeletonize

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})

print("numpy", np.__version__, "| opencv", cv2.__version__, "| skimage", skimage.__version__,
      "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
print("cpus:", os.cpu_count())

## 2. Configuration

Everything the notebook needs is in one dataclass, exactly as in the baselines. Three knobs are
worth knowing about:

- `NATIVE_IMAGE_STATS` — decode the full-resolution RGB tiles as well as the masks. Doubles the
  cost of section 6 but is the only way to report sharpness and exposure at the **native** GSD,
  which is what a reviewer will assume you measured. Leave it on for the run whose numbers go in
  the paper.
- `PATCH_STATS` — the per-class connected-component and skeleton pass (sections 13-14). It is the
  slowest pure-CPU block, ~2-4 min for 408 tiles.
- `PHASH_THRESHOLD` — Hamming distance below which two tiles are flagged as near-duplicates in
  section 17. 8/64 is the conventional cut for perceptual hashes.

`TILE_SIDE_M` is the one number here that is **an assumption, not a measurement**: the Atlas
coverage layer is gone, so the 300 m tile side comes from the export configuration, not from the
files. Everything downstream that is expressed in metres inherits that assumption, so it is stated
explicitly in the paper rather than buried.

In [ ]:
@dataclass
class CFG:
    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    IMG_SIZE: int = 512                 # working resolution used by every model in part 1
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255

    # ---- study-area geometry (from the QGIS Atlas export configuration) ----
    TILE_SIDE_M: float = 300.0          # ASSUMPTION: square tile footprint, no overlap

    # ---- split protocol (mirrors the baselines, for section 18 only) ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2
    SEED: int = 42

    # ---- analysis knobs ----
    NATIVE_IMAGE_STATS: bool = True     # decode full-res RGB too (sharpness/exposure at native GSD)
    PATCH_STATS: bool = True            # connected components + skeletons (sections 13-14)
    SPECTRAL_PX_PER_TILE: int = 400     # pixels sampled per class per tile for section 12
    PHASH_THRESHOLD: int = 8            # Hamming distance for the near-duplicate flag
    ROLL_WINDOW: int = 25               # rolling window over export order in section 19

    # ---- runtime / output ----
    NUM_WORKERS: int = 4
    CACHE_NPZ: str = "/kaggle/working/dsstats_cache.npz"
    OUT_DIR: str = "/kaggle/working/dataset_stats"
    FIG_DIR: str = "/kaggle/working/dataset_stats/figs"

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.FIG_DIR, exist_ok=True)
random.seed(cfg.SEED); np.random.seed(cfg.SEED)

STATS: Dict[str, object] = {}      # everything printed also lands here and is dumped in section 21
LATEX: Dict[str, str] = {}         # ready-to-paste table bodies

print("DATA_ROOT:", cfg.DATA_ROOT)
print("outputs  :", cfg.OUT_DIR)
print(f"knobs: native_img={cfg.NATIVE_IMAGE_STATS} patch={cfg.PATCH_STATS} "
      f"tile_side={cfg.TILE_SIDE_M} m")

## 3. File inventory and filename parsing

Same regex-based grouping as the training notebooks — `image{N}.png` / `image{N}hf.png` /
`image{N}vf.png`, masks mirroring with the `mask` prefix, `N` ascending in export order **with
gaps** where mono-class tiles were dropped after export.

Two things are measured here that the training notebooks never needed:

1. **On-disk geometry of every file**, read from the PNG header via PIL rather than by decoding.
   The paper claims a uniform 2345x2351 RGB raster; this verifies it over all 2448 files instead
   of over the handful anyone has actually opened. Any tile with a different size or a palette /
   RGBA mode would silently change its GSD, so this is a correctness check, not trivia.
2. **The gap structure of the export sequence.** The number of dropped ids, and whether they fall
   in long runs or singletons, is what tells you whether "export order" is a usable spatial proxy
   at all. Long contiguous drop-runs mean the retained tiles are not a connected sweep.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)
ORIG = [g for i in BASE_IDS for g in groups[i] if g[1] == "orig"]     # the 408 independent tiles
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert len(ORIG) == 408
assert all(sorted(v[1] for v in groups[i]) == ["hf", "orig", "vf"] for i in BASE_IDS)

# ---- header-only geometry probe over all 2448 files ----
def probe(p):
    with Image.open(p) as im:
        return im.size, im.mode
shapes, modes, bytes_img, bytes_msk = {}, {}, 0, 0
for _, _, iname, mname in records:
    for d, nm in ((IMG_DIR, iname), (MASK_DIR, mname)):
        (w, h), mo = probe(d / nm)
        shapes[(w, h)] = shapes.get((w, h), 0) + 1
        modes[mo] = modes.get(mo, 0) + 1
    bytes_img += (IMG_DIR / iname).stat().st_size
    bytes_msk += (MASK_DIR / mname).stat().st_size

print(f"{len(records)} image files + {len(records)} mask files | {len(BASE_IDS)} base tiles")
print("raster sizes (w,h) -> count:", shapes)
print("PIL modes          -> count:", modes)
print(f"on-disk size: images {bytes_img/1e9:.2f} GB + masks {bytes_msk/1e9:.2f} GB "
      f"= {(bytes_img+bytes_msk)/1e9:.2f} GB")

assert len(shapes) == 1, "tiles are NOT uniformly sized -- GSD is not constant, stop and check"
NAT_W, NAT_H = next(iter(shapes))

# ---- export-sequence gap structure ----
missing = sorted(set(range(BASE_IDS[0], BASE_IDS[-1] + 1)) - set(BASE_IDS))
runs, cur = [], []
for m_ in missing:
    if cur and m_ == cur[-1] + 1: cur.append(m_)
    else:
        if cur: runs.append(cur)
        cur = [m_]
if cur: runs.append(cur)
run_len = [len(r) for r in runs]

print(f"\nid range {BASE_IDS[0]}..{BASE_IDS[-1]} | exported ids {BASE_IDS[-1]-BASE_IDS[0]+1} | "
      f"retained {len(BASE_IDS)} | dropped {len(missing)} ({len(missing)/(BASE_IDS[-1]-BASE_IDS[0]+1)*100:.1f}%)")
print(f"dropped ids form {len(runs)} runs | longest run {max(run_len) if run_len else 0} | "
      f"singletons {sum(1 for r in run_len if r == 1)}")

STATS["inventory"] = {
    "n_files_total": 2 * len(records), "n_image_files": len(records), "n_pairs": len(records),
    "n_base_tiles": len(BASE_IDS), "n_variants_per_tile": 3,
    "native_width": NAT_W, "native_height": NAT_H, "pil_modes": modes,
    "size_images_gb": bytes_img / 1e9, "size_masks_gb": bytes_msk / 1e9,
    "size_total_gb": (bytes_img + bytes_msk) / 1e9,
    "id_min": BASE_IDS[0], "id_max": BASE_IDS[-1],
    "n_exported_ids": BASE_IDS[-1] - BASE_IDS[0] + 1,
    "n_dropped_ids": len(missing), "n_drop_runs": len(runs),
    "longest_drop_run": int(max(run_len)) if run_len else 0,
}

## 4. Palette definition and the channel-order guard

The class table is in **BGR**, and `cv2.imread` returns BGR, so the keys apply directly. The trap
is unchanged from the training notebooks: `(128,0,0)` = Residential and `(0,0,128)` = River are
exact reverses, so a silent RGB/BGR flip swaps the most common class with the rarest one and every
number in Section III would be wrong in a way that still looks plausible.

The lookup is a 16 MB `uint32 -> uint8` table so that mapping a 5.5 MP mask is one vectorized
gather rather than six boolean comparisons. Default value is 255 (ignore), which means **any colour
not in the table is counted as unlabelled** — section 7 reports exactly how much of the dataset
that is, over every mask rather than a sample.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}
CLASS_TO_BGR = {v: k for k, v in BGR_TO_CLASS.items()}

def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]
PALETTE_RGB[255] = (0, 0, 0)
def colorize(lab): return PALETTE_RGB[lab]

# ---- guard on a small random sample before committing to the 12-minute pass ----
rs = np.random.RandomState(cfg.SEED)
probe_recs = [ORIG[i] for i in rs.choice(len(ORIG), 12, replace=False)]
guard = np.zeros(7, np.int64)
for _, _, _, mname in probe_recs:
    lab = LUT[pack_bgr(cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR))]
    guard += np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)
tot = guard.sum()
for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):
    print(f"  {nm:<14}{guard[i]/tot*100:7.2f}%")
assert guard[0] > guard[2] * 3, "Residential <= 3x River -- channel order is inverted. STOP."
print("\nchannel-order check PASSED")

## 5. Study-area geometry and ground sampling distance

Pure arithmetic, but these are the numbers Section III opens with and they are easy to get wrong.

- **Native GSD** = tile side / raster width. A 300 m tile rendered to 2345 px is ~0.128 m/px,
  which is finer than any public satellite product — it is the resolution of the *basemap render*,
  not of an underlying sensor. That distinction has to be stated in the paper: the effective
  optical resolution is whatever the imagery provider's source is, and the raster is an upsampled
  rendering of it. Reviewers of remote-sensing papers check this.
- **Working GSD** = tile side / 512 ~ 0.586 m/px. Every metric in the results section is computed
  at this scale, so all metre-denominated structure statistics below use it.
- **Annotated area** = retained tiles x tile footprint. This is the honest coverage number.
  **Swept extent** = exported ids x footprint, which is larger, and the difference is the tiles
  dropped for being mono-class. Report both, and say which is which.

In [ ]:
GSD_NATIVE = cfg.TILE_SIDE_M / NAT_W
GSD_WORK   = cfg.TILE_SIDE_M / cfg.IMG_SIZE
TILE_AREA_M2 = cfg.TILE_SIDE_M ** 2
TILE_AREA_HA = TILE_AREA_M2 / 1e4
ANNOT_KM2 = len(BASE_IDS) * TILE_AREA_M2 / 1e6
SWEPT_KM2 = STATS["inventory"]["n_exported_ids"] * TILE_AREA_M2 / 1e6

print(f"tile footprint      : {cfg.TILE_SIDE_M:.0f} x {cfg.TILE_SIDE_M:.0f} m "
      f"= {TILE_AREA_HA:.2f} ha")
print(f"native raster       : {NAT_W} x {NAT_H} px  -> GSD {GSD_NATIVE:.4f} m/px "
      f"({GSD_NATIVE*100:.2f} cm/px)")
print(f"working raster      : {cfg.IMG_SIZE} x {cfg.IMG_SIZE} px -> GSD {GSD_WORK:.4f} m/px")
print(f"downsample factor   : {NAT_W/cfg.IMG_SIZE:.2f}x  (area {(NAT_W/cfg.IMG_SIZE)**2:.1f}x fewer px)")
print(f"annotated coverage  : {len(BASE_IDS)} tiles = {ANNOT_KM2:.2f} km^2")
print(f"swept extent        : {STATS['inventory']['n_exported_ids']} exported ids = {SWEPT_KM2:.2f} km^2")
print(f"aspect check        : {NAT_W}/{NAT_H} = {NAT_W/NAT_H:.4f} "
      f"({'square-ish, footprint assumption holds' if abs(NAT_W/NAT_H-1)<0.02 else 'NOT square -- footprint is not 300x300'})")

STATS["geometry"] = {
    "tile_side_m": cfg.TILE_SIDE_M, "tile_area_ha": TILE_AREA_HA,
    "gsd_native_m": GSD_NATIVE, "gsd_working_m": GSD_WORK,
    "downsample_factor": NAT_W / cfg.IMG_SIZE,
    "annotated_area_km2": ANNOT_KM2, "swept_extent_km2": SWEPT_KM2,
}

## 6. The single full-resolution pass

One `ProcessPoolExecutor` sweep over the **408 original tiles**. Per tile it returns:

- the full set of distinct packed BGR values in the mask with their counts (feeds section 7);
- the class histogram at **full resolution** (the ground truth for every share reported);
- the class histogram after `INTER_NEAREST` downsampling to 512 (feeds section 8);
- the 512 label array and, when `NATIVE_IMAGE_STATS` is on, the 512 RGB array and native-resolution
  image statistics (per-channel sums, Laplacian variance, clipped-pixel counts).

Full-resolution counts are computed **before** any resize, which is the whole point — quoting Road
at 2.28 % from a downsampled raster and calling it the dataset's class balance would be wrong by
whatever nearest-neighbour resampling does to a 9-pixel-wide structure. Section 8 quantifies that
gap instead of assuming it away.

Everything lands in `dsstats_cache.npz`, so re-running the notebook is instant. Delete that file if
the dataset ever changes (it will not — it is frozen).

In [ ]:
S = cfg.IMG_SIZE

def _pass_one(rec):
    base_id, variant, iname, mname = rec
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    packed = pack_bgr(msk)
    u, c = np.unique(packed, return_counts=True)
    lab = LUT[packed]
    cnt_full = np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)
    lab512 = cv2.resize(lab, (S, S), interpolation=cv2.INTER_NEAREST)
    cnt512 = np.bincount(np.where(lab512 == 255, 6, lab512).ravel(), minlength=7)

    out = {"base_id": base_id, "iname": iname, "colours": (u.tolist(), c.tolist()),
           "cnt_full": cnt_full, "cnt512": cnt512, "lab512": lab512}

    if cfg.NATIVE_IMAGE_STATS:
        img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
        f = img.reshape(-1, 3).astype(np.float64)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        out.update({
            "px": f.shape[0], "sum": f.sum(0), "sumsq": (f ** 2).sum(0),
            "lapvar": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
            "sat_mean": float(hsv[..., 1].mean()),
            "clip_hi": int((gray >= 250).sum()), "clip_lo": int((gray <= 5).sum()),
            "img512": cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA),
        })
    return out

CACHE = Path(cfg.CACHE_NPZ)
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    CNT_FULL, CNT512 = z["cnt_full"], z["cnt512"]
    LAB512, IMG512 = z["lab512"], (z["img512"] if "img512" in z else None)
    COLOUR_HITS = z["colour_hits"].item()
    IMGSTAT = z["imgstat"].item()
    ORDER = z["order"].tolist()
    print(f"loaded cache {CACHE} | tiles {len(ORDER)}")
else:
    t0 = time.time()
    CNT_FULL, CNT512, LAB512, IMG512, ORDER = [], [], [], [], []
    COLOUR_HITS: Dict[int, int] = {}
    IMGSTAT = {"px": 0.0, "sum": np.zeros(3), "sumsq": np.zeros(3),
               "lapvar": [], "sat": [], "clip_hi": 0, "clip_lo": 0}
    with ProcessPoolExecutor(max_workers=cfg.NUM_WORKERS) as ex:
        for k, r in enumerate(ex.map(_pass_one, ORIG, chunksize=4)):
            ORDER.append(r["base_id"])
            CNT_FULL.append(r["cnt_full"]); CNT512.append(r["cnt512"]); LAB512.append(r["lab512"])
            for uu, cc in zip(*r["colours"]):
                COLOUR_HITS[uu] = COLOUR_HITS.get(uu, 0) + cc
            if cfg.NATIVE_IMAGE_STATS:
                IMG512.append(r["img512"])
                IMGSTAT["px"] += r["px"]; IMGSTAT["sum"] += r["sum"]; IMGSTAT["sumsq"] += r["sumsq"]
                IMGSTAT["lapvar"].append(r["lapvar"]); IMGSTAT["sat"].append(r["sat_mean"])
                IMGSTAT["clip_hi"] += r["clip_hi"]; IMGSTAT["clip_lo"] += r["clip_lo"]
            if (k + 1) % 50 == 0:
                el = time.time() - t0
                print(f"  {k+1}/{len(ORIG)}  {el:.0f}s  eta {el/(k+1)*(len(ORIG)-k-1):.0f}s")
    CNT_FULL = np.stack(CNT_FULL); CNT512 = np.stack(CNT512); LAB512 = np.stack(LAB512)
    IMG512 = np.stack(IMG512) if cfg.NATIVE_IMAGE_STATS else None
    np.savez_compressed(CACHE, cnt_full=CNT_FULL, cnt512=CNT512, lab512=LAB512,
                        order=np.array(ORDER), colour_hits=COLOUR_HITS, imgstat=IMGSTAT,
                        **({"img512": IMG512} if IMG512 is not None else {}))
    print(f"pass complete in {time.time()-t0:.0f}s -> {CACHE}")

assert ORDER == BASE_IDS, "cache tile order does not match the current file listing"
print("arrays:", CNT_FULL.shape, CNT512.shape, LAB512.shape,
      IMG512.shape if IMG512 is not None else "(no img512)")

## 7. Exhaustive palette audit

Over **every pixel of every original mask** — 408 x 5.5 M = 2.25 x 10^9 label pixels — not a
sample. Three questions get answered:

1. Does any colour outside the six-entry palette appear? Anti-aliased polygon edges, JPEG
   contamination, or an accidental palette PNG would show up here as a long tail of near-miss
   colours, and each of those pixels is silently training on `ignore`.
2. What is the true unlabelled share? The paper quotes ~0.07 %; this is the exhaustive value.
3. How many tiles contain any unmapped pixel at all?

If the unmapped share is dominated by exactly `(0,0,0)`, the labels are clean and "ignore" simply
means "not annotated". If it is spread over dozens of near-palette colours, the mask export
resampled somewhere and that needs a sentence in the limitations paragraph.

In [ ]:
known_packed = {int(pack_bgr(np.array(b, np.uint8).reshape(1, 1, 3))[0, 0]): i
                for b, i in BGR_TO_CLASS.items()}
tot_px = sum(COLOUR_HITS.values())
rows = []
for k, n in sorted(COLOUR_HITS.items(), key=lambda kv: -kv[1]):
    b, g, r = (k >> 16) & 255, (k >> 8) & 255, k & 255
    cls = known_packed.get(k)
    rows.append({"B": b, "G": g, "R": r, "pixels": n, "share_pct": n / tot_px * 100,
                 "class": cfg.CLASS_NAMES[cls] if cls is not None else "<unmapped -> ignore>"})
cdf = pd.DataFrame(rows)
print(f"distinct BGR colours across all {len(ORIG)} masks: {len(cdf)}")
print(cdf.head(15).to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

unmapped = cdf[cdf["class"].str.startswith("<")]
print(f"\nunmapped colours: {len(unmapped)} | pixels {unmapped['pixels'].sum():,} "
      f"({unmapped['share_pct'].sum():.4f}% of all label pixels)")
if len(unmapped):
    print("top unmapped:")
    print(unmapped.head(8).to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

n_tiles_with_ignore = int((CNT_FULL[:, 6] > 0).sum())
print(f"\ntiles containing >=1 unlabelled pixel: {n_tiles_with_ignore}/{len(ORIG)} "
      f"({n_tiles_with_ignore/len(ORIG)*100:.1f}%)")
print(f"max per-tile unlabelled share: {(CNT_FULL[:,6]/CNT_FULL.sum(1)).max()*100:.3f}%")
cdf.to_csv(f"{cfg.OUT_DIR}/palette_audit.csv", index=False)

STATS["palette"] = {
    "n_distinct_colours": int(len(cdf)), "n_unmapped_colours": int(len(unmapped)),
    "unmapped_pixel_share_pct": float(unmapped["share_pct"].sum()),
    "tiles_with_ignore": n_tiles_with_ignore,
    "max_tile_ignore_share_pct": float((CNT_FULL[:, 6] / CNT_FULL.sum(1)).max() * 100),
    "total_label_pixels": int(tot_px),
}

## 8. Resampling fidelity: 2345 -> 512

The models never see the native raster; they see a 4.58x nearest-neighbour downsample of the
labels. For a class that is 9 px wide at 512 this is a survivable but not free operation, and for
River — 0.57 % of pixels, thin and sinuous — it is exactly the kind of class that can lose a
measurable fraction of its area.

Two numbers per class:

- **share drift** — class share at 512 minus class share at full resolution, in percentage points
  and relative terms. This tells you whether the class balance quoted in the paper is the same at
  both scales (it should be quoted at the resolution actually used for training).
- **presence loss** — tiles where a class exists at full resolution but has zero pixels at 512.
  Any non-zero value here means some annotation is being deleted outright by the downsample, which
  is a limitation worth one sentence.

In [ ]:
sf, s5 = CNT_FULL.sum(0), CNT512.sum(0)
pf, p5 = sf / sf.sum() * 100, s5 / s5.sum() * 100
names = list(cfg.CLASS_NAMES) + ["<ignore>"]

rows = []
for i, nm in enumerate(names):
    lost = int(((CNT_FULL[:, i] > 0) & (CNT512[:, i] == 0)).sum())
    rows.append({"class": nm, "share_fullres_pct": pf[i], "share_512_pct": p5[i],
                 "abs_drift_pp": p5[i] - pf[i],
                 "rel_drift_pct": (p5[i] - pf[i]) / pf[i] * 100 if pf[i] > 0 else 0.0,
                 "tiles_present_fullres": int((CNT_FULL[:, i] > 0).sum()),
                 "tiles_lost_at_512": lost})
rdf = pd.DataFrame(rows)
print(rdf.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print(f"\ntotal label pixels: full-res {sf.sum():,} -> 512 {s5.sum():,}")
rdf.to_csv(f"{cfg.OUT_DIR}/resampling_fidelity.csv", index=False)
STATS["resampling"] = rdf.to_dict("records")

## 9. Class distribution

The headline imbalance table. Reported at the **working resolution** (512), because that is what
every model in Section VI was trained and evaluated on, with the full-resolution column kept
alongside so the two can never be confused.

Beyond raw shares:
- **imbalance ratio** = most common share / least common share. A single number for the abstract.
- **area in hectares**, obtained from pixel counts x working GSD^2. Percentages are what the model
  sees; hectares are what a planner sees, and Section III should give both.
- **effective number of classes** = exp(Shannon entropy of the class distribution). With 6 nominal
  classes, a value near 4 says the problem behaves like a 4-class problem with two tokens attached.

In [ ]:
px_area_m2 = GSD_WORK ** 2
rows = []
for i, nm in enumerate(names):
    ha = s5[i] * px_area_m2 / 1e4
    rows.append({"idx": i if i < 6 else 255, "class": nm,
                 "pixels_512": int(s5[i]), "share_pct": p5[i],
                 "area_ha": ha, "area_km2": ha / 100,
                 "share_fullres_pct": pf[i]})
ddf = pd.DataFrame(rows)
print(ddf.to_string(index=False, float_format=lambda v: f"{v:10.3f}"))

q = p5[:6] / p5[:6].sum()
H = float(-(q * np.log(q + 1e-12)).sum())
IR = float(p5[:6].max() / p5[:6].min())
print(f"\nimbalance ratio (max/min over 6 classes): {IR:.1f}x")
print(f"Shannon entropy {H:.4f} nats | effective number of classes exp(H) = {math.exp(H):.2f} of 6")
print(f"Gini-Simpson 1 - sum p^2 = {1 - float((q**2).sum()):.4f}")
ddf.to_csv(f"{cfg.OUT_DIR}/class_summary.csv", index=False)

LATEX["class_distribution"] = "\n".join(
    f"{r['class']} & {r['share_pct']:.2f} & {r['area_ha']:.1f} & {r['share_fullres_pct']:.2f} \\\\"
    for _, r in ddf.iterrows())
STATS["class_distribution"] = {"records": ddf.to_dict("records"), "imbalance_ratio": IR,
                               "entropy_nats": H, "effective_classes": math.exp(H)}

## 10. Per-tile class statistics

Dataset-level shares hide the thing that actually governs training: **how the classes are
distributed across tiles**. A class at 2 % of pixels concentrated in 30 tiles is a much harder
problem than the same 2 % spread evenly over 408, because most minibatches contain none of it.

Computed here:
- tiles containing each class, at all and above a 1 % share (the "meaningfully present" count);
- the distribution of classes-per-tile — how many tiles are effectively 2-class scenes;
- per-tile share matrix written to CSV, which every later section reuses.

In [ ]:
SHARE = CNT512[:, :6] / CNT512[:, :6].sum(1, keepdims=True)      # 408 x 6, ignore excluded
PRESENT = CNT512[:, :6] > 0
MEANING = SHARE >= 0.01

rows = []
for i, nm in enumerate(cfg.CLASS_NAMES):
    s = SHARE[:, i]
    rows.append({"class": nm,
                 "tiles_present": int(PRESENT[:, i].sum()),
                 "pct_tiles_present": PRESENT[:, i].mean() * 100,
                 "tiles_ge_1pct": int(MEANING[:, i].sum()),
                 "mean_share_pct": s.mean() * 100,
                 "median_share_pct": np.median(s) * 100,
                 "p90_share_pct": np.percentile(s, 90) * 100,
                 "max_share_pct": s.max() * 100})
tdf = pd.DataFrame(rows)
print(tdf.to_string(index=False, float_format=lambda v: f"{v:9.2f}"))

npc = PRESENT.sum(1)
print("\nclasses present per tile:")
for k in range(1, 7):
    print(f"  {k} classes: {int((npc==k).sum()):4d} tiles ({(npc==k).mean()*100:5.1f}%)")
print(f"mean classes per tile: {npc.mean():.2f}")

per_tile = pd.DataFrame(SHARE * 100, columns=[f"share_{n}" for n in cfg.CLASS_NAMES])
per_tile.insert(0, "base_id", BASE_IDS)
per_tile["n_classes"] = npc
per_tile["ignore_pct"] = CNT512[:, 6] / CNT512.sum(1) * 100
per_tile.to_csv(f"{cfg.OUT_DIR}/per_tile_stats.csv", index=False)
tdf.to_csv(f"{cfg.OUT_DIR}/per_tile_class_summary.csv", index=False)
STATS["per_tile"] = {"records": tdf.to_dict("records"),
                     "classes_per_tile_mean": float(npc.mean()),
                     "classes_per_tile_hist": {int(k): int((npc == k).sum()) for k in range(1, 7)}}

## 11. Class co-occurrence

Which classes share a tile. Two views of the same 6x6 matrix:

- **conditional presence** P(column present | row present), which is what a sampler experiences;
- **Jaccard over tiles**, symmetric, which is the cleaner number for a figure.

This is where the "River is nearly invisible" claim gets teeth: if River co-occurs with Residential
in almost every tile it appears in, then a Residential-dominated loss will always be competing
directly against it, and the class-weighting discussion in Section IV has a concrete justification.

In [ ]:
P = PRESENT.astype(np.float64)
inter = P.T @ P
cond = inter / np.maximum(P.sum(0)[:, None], 1)
union = P.sum(0)[:, None] + P.sum(0)[None, :] - inter
jac = inter / np.maximum(union, 1)

def show(mat, title):
    print(f"\n{title}")
    print(" " * 14 + "".join(f"{n[:9]:>11}" for n in cfg.CLASS_NAMES))
    for i, nm in enumerate(cfg.CLASS_NAMES):
        print(f"{nm:<14}" + "".join(f"{mat[i,j]:11.3f}" for j in range(6)))

show(cond, "P(col present | row present)")
show(jac, "tile-level Jaccard")
np.savetxt(f"{cfg.OUT_DIR}/cooccurrence_conditional.csv", cond, delimiter=",", fmt="%.6f")
np.savetxt(f"{cfg.OUT_DIR}/cooccurrence_jaccard.csv", jac, delimiter=",", fmt="%.6f")
STATS["cooccurrence"] = {"conditional": cond.tolist(), "jaccard": jac.tolist()}

## 12. Spectral statistics and class separability

The two confusions reported in Section VII — Unused Land vs Agricultural, and urban trees vs
Forest — are usually described qualitatively. This section measures them.

Per class, RGB pixels are sampled from the 512 imagery (up to `SPECTRAL_PX_PER_TILE` per class per
tile) and summarised as a Gaussian. Pairwise separability is then the **Jeffries-Matusita
distance**, derived from the Bhattacharyya distance between those Gaussians:

`B = 1/8 (m1-m2)' S^-1 (m1-m2) + 1/2 ln( |S| / sqrt(|S1||S2|) )`,  `S = (S1+S2)/2`,
`JM = 2(1 - exp(-B))`, range 0-2.

The convention in the remote-sensing literature: **JM > 1.9 = well separated, 1.0-1.9 = partially
separable, < 1.0 = spectrally inseparable.** A JM below 1 for a class pair is a direct,
citable statement that the confusion is not a model failure but a property of three-band RGB
imagery — which is exactly the framing Section VII needs. Note the honest caveat: this is RGB only,
with no NIR, so low separability for vegetation classes is expected and should be stated as a
dataset limitation.

In [ ]:
assert IMG512 is not None, "set NATIVE_IMAGE_STATS=True and delete the npz cache to get imagery"
rs = np.random.RandomState(cfg.SEED)
samples = {i: [] for i in range(6)}
for t in range(len(ORDER)):
    lab, img = LAB512[t], IMG512[t]
    rgb = np.ascontiguousarray(img[..., ::-1]).reshape(-1, 3)
    fl = lab.ravel()
    for c in range(6):
        idx = np.flatnonzero(fl == c)
        if idx.size == 0: continue
        if idx.size > cfg.SPECTRAL_PX_PER_TILE:
            idx = rs.choice(idx, cfg.SPECTRAL_PX_PER_TILE, replace=False)
        samples[c].append(rgb[idx])
SAMP = {c: np.concatenate(v).astype(np.float64) for c, v in samples.items() if v}

rows = []
for c, v in SAMP.items():
    m, s = v.mean(0), v.std(0)
    rows.append({"class": cfg.CLASS_NAMES[c], "n_px": len(v),
                 "R_mean": m[0], "G_mean": m[1], "B_mean": m[2],
                 "R_std": s[0], "G_std": s[1], "B_std": s[2],
                 "brightness": m.mean()})
sdf = pd.DataFrame(rows)
print(sdf.to_string(index=False, float_format=lambda v: f"{v:9.2f}"))

MU = {c: v.mean(0) for c, v in SAMP.items()}
CV = {c: np.cov(v, rowvar=False) + np.eye(3) * 1e-6 for c, v in SAMP.items()}
JM = np.zeros((6, 6))
for a in range(6):
    for b in range(6):
        if a == b or a not in MU or b not in MU: continue
        d = MU[a] - MU[b]; S = (CV[a] + CV[b]) / 2
        B = 0.125 * d @ np.linalg.inv(S) @ d + 0.5 * np.log(
            np.linalg.det(S) / np.sqrt(np.linalg.det(CV[a]) * np.linalg.det(CV[b])))
        JM[a, b] = 2 * (1 - np.exp(-B))

print("\nJeffries-Matusita separability (0-2; >1.9 separable, <1.0 inseparable)")
print(" " * 14 + "".join(f"{n[:9]:>11}" for n in cfg.CLASS_NAMES))
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}" + "".join(f"{JM[i,j]:11.3f}" for j in range(6)))

pairs = sorted(((JM[a, b], cfg.CLASS_NAMES[a], cfg.CLASS_NAMES[b])
                for a in range(6) for b in range(a + 1, 6)))
print("\nleast separable pairs:")
for v, a, b in pairs[:5]:
    print(f"  {a:<14} vs {b:<14} JM = {v:.3f}")

sdf.to_csv(f"{cfg.OUT_DIR}/spectral_stats.csv", index=False)
np.savetxt(f"{cfg.OUT_DIR}/separability_jm.csv", JM, delimiter=",", fmt="%.6f")
STATS["spectral"] = {"per_class": sdf.to_dict("records"), "jm_matrix": JM.tolist(),
                     "least_separable": [[float(v), a, b] for v, a, b in pairs[:5]]}

## 13. Landscape and patch metrics

Standard landscape-ecology descriptors, computed per tile at 512 and averaged. They characterise
the *spatial* structure of the labels, which pixel shares cannot:

- **patch count / patch density (per 100 ha)** — 8-connected components per class;
- **mean and median patch area** in m^2, and the largest-patch index;
- **edge density (m/ha)** — length of class boundaries per unit area, the standard scalar for
  landscape fragmentation;
- **SHDI / SIDI** — Shannon and Simpson diversity per tile.

These are the same metrics used to compare real against generated layouts in the part-2 work, so
computing them here once means Section III and the generation paper quote a consistent set.

In [ ]:
def patch_metrics(lab):
    out = {}
    for c in range(6):
        m = (lab == c).astype(np.uint8)
        a = int(m.sum())
        if a == 0:
            out[c] = (0, 0.0, 0.0, 0.0); continue
        n, _, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        ar = st[1:, cv2.CC_STAT_AREA].astype(np.float64)
        out[c] = (int(n - 1), float(ar.mean()), float(np.median(ar)), float(ar.max() / a))
    return out

def edge_density_m_per_ha(lab):
    v = (lab[1:, :] != lab[:-1, :]).sum()
    h = (lab[:, 1:] != lab[:, :-1]).sum()
    return (v + h) * GSD_WORK / TILE_AREA_HA

if cfg.PATCH_STATS:
    t0 = time.time()
    npatch = np.zeros((len(ORDER), 6)); mpa = np.zeros((len(ORDER), 6))
    medpa = np.zeros((len(ORDER), 6)); lpi = np.zeros((len(ORDER), 6))
    ed = np.zeros(len(ORDER)); shdi = np.zeros(len(ORDER)); sidi = np.zeros(len(ORDER))
    for t in range(len(ORDER)):
        pm = patch_metrics(LAB512[t])
        for c in range(6):
            npatch[t, c], mpa[t, c], medpa[t, c], lpi[t, c] = pm[c]
        ed[t] = edge_density_m_per_ha(LAB512[t])
        p = SHARE[t]; p = p[p > 0]
        shdi[t] = -(p * np.log(p)).sum(); sidi[t] = 1 - (p ** 2).sum()
        if (t + 1) % 100 == 0: print(f"  {t+1}/{len(ORDER)}  {time.time()-t0:.0f}s")

    rows = []
    for c in range(6):
        pres = PRESENT[:, c]
        rows.append({"class": cfg.CLASS_NAMES[c],
                     "patches_per_tile": npatch[pres, c].mean() if pres.any() else 0,
                     "patch_density_per_100ha": npatch[pres, c].mean() / TILE_AREA_HA * 100 if pres.any() else 0,
                     "mean_patch_area_m2": (mpa[pres, c].mean() * GSD_WORK ** 2) if pres.any() else 0,
                     "median_patch_area_m2": (medpa[pres, c].mean() * GSD_WORK ** 2) if pres.any() else 0,
                     "largest_patch_index": lpi[pres, c].mean() if pres.any() else 0})
    ldf = pd.DataFrame(rows)
    print(ldf.to_string(index=False, float_format=lambda v: f"{v:12.2f}"))
    print(f"\nedge density  : {ed.mean():8.1f} +/- {ed.std():.1f} m/ha")
    print(f"SHDI (Shannon): {shdi.mean():8.4f} +/- {shdi.std():.4f}")
    print(f"SIDI (Simpson): {sidi.mean():8.4f} +/- {sidi.std():.4f}")
    ldf.to_csv(f"{cfg.OUT_DIR}/landscape_metrics.csv", index=False)
    STATS["landscape"] = {"per_class": ldf.to_dict("records"),
                          "edge_density_m_per_ha_mean": float(ed.mean()),
                          "edge_density_m_per_ha_std": float(ed.std()),
                          "shdi_mean": float(shdi.mean()), "sidi_mean": float(sidi.mean())}
else:
    print("PATCH_STATS off -- skipped")

## 14. Thin-structure geometry — Road and River

The two classes the paper singles out as hard are hard for a geometric reason, and this quantifies
it in metres rather than in pixels.

Method per tile: binarise the class, skeletonise it, and read the Euclidean distance transform
**along the skeleton**. Twice that distance is the local width, so the median over all skeleton
pixels is a robust width estimate; the skeleton pixel count times the working GSD approximates
centreline length. Two caveats, both worth one clause in the paper: the length estimate ignores the
sqrt(2) correction for diagonal skeleton steps and so slightly under-estimates, and widths are
measured after the downsample to 512, which is the resolution the model actually operates at.

The derived numbers — median width in metres and road length density in km/km^2 — are directly
comparable to published cadastral figures for Kathmandu, which is a genuinely useful sanity check
on the annotation and a good sentence for Section III.

In [ ]:
if cfg.PATCH_STATS:
    thin = {}
    for c, nm in [(1, "Road"), (2, "River")]:
        widths, lengths, ntiles = [], 0.0, 0
        for t in range(len(ORDER)):
            m = (LAB512[t] == c)
            if m.sum() < 20: continue
            ntiles += 1
            sk = skeletonize(m)
            if sk.sum() == 0: continue
            dt = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 5)
            widths.append(2.0 * dt[sk])
            lengths += float(sk.sum()) * GSD_WORK
        w = np.concatenate(widths) if widths else np.array([0.0])
        thin[nm] = {
            "tiles_scored": ntiles,
            "median_width_px": float(np.median(w)),
            "median_width_m": float(np.median(w) * GSD_WORK),
            "p10_width_m": float(np.percentile(w, 10) * GSD_WORK),
            "p90_width_m": float(np.percentile(w, 90) * GSD_WORK),
            "total_length_km": lengths / 1000,
            "length_density_km_per_km2": (lengths / 1000) / ANNOT_KM2,
            "px_per_width_at_512": float(np.median(w)),
        }
        d = thin[nm]
        print(f"{nm}")
        print(f"  tiles with the class      : {d['tiles_scored']}/{len(ORDER)}")
        print(f"  median width              : {d['median_width_px']:.1f} px at 512 "
              f"= {d['median_width_m']:.2f} m")
        print(f"  width p10-p90             : {d['p10_width_m']:.2f} - {d['p90_width_m']:.2f} m")
        print(f"  total centreline length   : {d['total_length_km']:.1f} km")
        print(f"  length density            : {d['length_density_km_per_km2']:.2f} km/km^2")
        print(f"  width in output stride-4 px (SegFormer H/4): "
              f"{d['median_width_px']/4:.2f} px  <- why thin classes vanish before upsampling")
    json.dump(thin, open(f"{cfg.OUT_DIR}/thin_structures.json", "w"), indent=2)
    STATS["thin_structures"] = thin
else:
    print("PATCH_STATS off -- skipped")

## 15. Imagery statistics

Radiometric description of the RGB tiles, measured at **native** resolution during the section-6
pass (so it is not contaminated by the INTER_AREA downsample, which suppresses high-frequency
content and would inflate apparent smoothness).

- per-channel mean and standard deviation in 0-255 and normalised to 0-1. If these differ
  substantially from the ImageNet statistics the models normalise with, that is worth a sentence
  in Section V, since every backbone was pretrained under the ImageNet assumption;
- mean HSV saturation, a compact proxy for how vivid the basemap rendering is;
- clipped-pixel fractions at both ends — evidence of tone-mapping in the basemap rather than raw
  radiometry;
- Laplacian variance per tile as a sharpness proxy, with the low tail listed so cloud, haze or
  blur-affected tiles can be inspected before they are described as clean.

In [ ]:
if cfg.NATIVE_IMAGE_STATS:
    n = IMGSTAT["px"]
    mean_bgr = IMGSTAT["sum"] / n
    var_bgr = IMGSTAT["sumsq"] / n - mean_bgr ** 2
    std_bgr = np.sqrt(np.maximum(var_bgr, 0))
    mean_rgb, std_rgb = mean_bgr[::-1], std_bgr[::-1]
    lap = np.array(IMGSTAT["lapvar"]); sat = np.array(IMGSTAT["sat"])

    print("per-channel statistics at native resolution (R, G, B):")
    print(f"  mean 0-255 : {mean_rgb[0]:7.2f} {mean_rgb[1]:7.2f} {mean_rgb[2]:7.2f}")
    print(f"  std  0-255 : {std_rgb[0]:7.2f} {std_rgb[1]:7.2f} {std_rgb[2]:7.2f}")
    print(f"  mean 0-1   : {mean_rgb[0]/255:7.4f} {mean_rgb[1]/255:7.4f} {mean_rgb[2]/255:7.4f}")
    print(f"  std  0-1   : {std_rgb[0]/255:7.4f} {std_rgb[1]/255:7.4f} {std_rgb[2]/255:7.4f}")
    print(f"  ImageNet   :  0.4850  0.4560  0.4060 / 0.2290 0.2240 0.2250 (for reference)")
    print(f"\nmean HSV saturation : {sat.mean():.2f} +/- {sat.std():.2f} (0-255)")
    print(f"clipped highlights  : {IMGSTAT['clip_hi']/n*100:.4f}% of pixels >= 250")
    print(f"clipped shadows     : {IMGSTAT['clip_lo']/n*100:.4f}% of pixels <= 5")
    print(f"\nsharpness (Laplacian variance, native res):")
    print(f"  mean {lap.mean():.1f} | median {np.median(lap):.1f} | "
          f"p5 {np.percentile(lap,5):.1f} | p95 {np.percentile(lap,95):.1f}")
    lo = np.argsort(lap)[:8]
    print("  8 least sharp tiles (inspect before calling the imagery uniform):")
    print("   ", [(int(BASE_IDS[i]), round(float(lap[i]), 1)) for i in lo])

    STATS["imagery"] = {
        "mean_rgb_255": mean_rgb.tolist(), "std_rgb_255": std_rgb.tolist(),
        "mean_rgb_01": (mean_rgb / 255).tolist(), "std_rgb_01": (std_rgb / 255).tolist(),
        "saturation_mean": float(sat.mean()),
        "clip_hi_pct": IMGSTAT["clip_hi"] / n * 100, "clip_lo_pct": IMGSTAT["clip_lo"] / n * 100,
        "laplacian_var_mean": float(lap.mean()), "laplacian_var_median": float(np.median(lap)),
        "least_sharp_tiles": [int(BASE_IDS[i]) for i in lo],
    }
else:
    print("NATIVE_IMAGE_STATS off -- skipped")

## 16. Flip-variant integrity

The dataset ships each tile three times: original, horizontal flip, vertical flip. Two claims in
the paper depend on that being literally true rather than approximately true:

1. statistics reported over 408 tiles generalise to the 1224 files unchanged;
2. the training notebooks are right to disable hflip/vflip augmentation, since re-applying them
   would produce duplicate views inside one epoch.

This verifies both by decoding a random subset of tiles and checking exact pixel equality between
`image{N}hf` and `flip(image{N})`, for images and masks. It also tells you **which axis** the `hf`
and `vf` tags actually correspond to, which is not something to assume from the letters.

In [ ]:
K = 24
rs = np.random.RandomState(cfg.SEED)
sel = [BASE_IDS[i] for i in rs.choice(len(BASE_IDS), K, replace=False)]
res = {"hf": {"axis1": 0, "axis0": 0, "neither": 0}, "vf": {"axis1": 0, "axis0": 0, "neither": 0}}
mask_ok = 0

for bid in sel:
    g = {v: (i, m) for _, v, i, m in groups[bid]}
    o_img = cv2.imread(str(IMG_DIR / g["orig"][0]), cv2.IMREAD_COLOR)
    o_msk = cv2.imread(str(MASK_DIR / g["orig"][1]), cv2.IMREAD_COLOR)
    for v in ("hf", "vf"):
        img = cv2.imread(str(IMG_DIR / g[v][0]), cv2.IMREAD_COLOR)
        msk = cv2.imread(str(MASK_DIR / g[v][1]), cv2.IMREAD_COLOR)
        if np.array_equal(img, o_img[:, ::-1]):
            res[v]["axis1"] += 1; ok = np.array_equal(msk, o_msk[:, ::-1])
        elif np.array_equal(img, o_img[::-1, :]):
            res[v]["axis0"] += 1; ok = np.array_equal(msk, o_msk[::-1, :])
        else:
            res[v]["neither"] += 1; ok = False
        mask_ok += int(ok)

print(f"checked {K} tiles x 2 variants")
for v in ("hf", "vf"):
    print(f"  {v}: left-right {res[v]['axis1']}/{K} | up-down {res[v]['axis0']}/{K} | "
          f"neither {res[v]['neither']}/{K}")
print(f"  mask flips consistent with image flips: {mask_ok}/{2*K}")
if res["hf"]["neither"] or res["vf"]["neither"]:
    print("  WARNING: some variants are not exact flips -- they are NOT redundant, "
          "and the 408-tile reporting basis needs revisiting")
else:
    print("  variants are exact mirrors -> 1224 files = 408 independent scenes. Confirmed.")
STATS["flip_integrity"] = {"n_checked": K, "result": res, "mask_consistent": mask_ok}

## 17. Near-duplicate detection

Adjacent tiles in a dense urban grid can look almost identical, and if two near-identical tiles land
on opposite sides of a train/val boundary the fold is leaky no matter how carefully the split was
constructed. Section VII already argues that the export-order block split is a *proxy* for spatial
separation; this puts a number on the residual risk.

A 64-bit perceptual hash (32x32 grey -> DCT -> 8x8 low-frequency block, median-thresholded) is
computed for each of the 408 tiles, and all 83 k pairs are compared. Reported: the number of pairs
below the threshold, whether those pairs are neighbours in export order (expected and harmless
within a block) or far apart (the leak-prone case), and the closest pairs so they can be eyeballed.

In [ ]:
def phash64(bgr):
    g = cv2.cvtColor(cv2.resize(bgr, (32, 32), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    d = cv2.dct(np.float32(g))[:8, :8].flatten()
    return (d > np.median(d[1:])).astype(np.uint8)

if IMG512 is not None:
    PH = np.stack([phash64(IMG512[t]) for t in range(len(ORDER))])
    D = (PH[:, None, :] != PH[None, :, :]).sum(-1)
    iu = np.triu_indices(len(ORDER), 1)
    dist = D[iu]
    near = np.flatnonzero(dist < cfg.PHASH_THRESHOLD)
    ids = np.array(BASE_IDS)
    pa, pb = ids[iu[0][near]], ids[iu[1][near]]
    seq_gap = np.abs(iu[0][near] - iu[1][near])

    print(f"pairs compared: {len(dist):,} | mean Hamming {dist.mean():.1f} | min {dist.min()}")
    print(f"near-duplicate pairs (< {cfg.PHASH_THRESHOLD}/64): {len(near)} "
          f"({len(near)/len(dist)*100:.4f}% of pairs)")
    if len(near):
        print(f"  of which adjacent in export order (gap <= 3): {(seq_gap<=3).sum()}")
        print(f"  of which far apart (gap > 20, the leak-prone case): {(seq_gap>20).sum()}")
        o = np.argsort(dist[near])[:10]
        print("  closest pairs (base_id_a, base_id_b, hamming, seq_gap):")
        for j in o:
            print(f"    {pa[j]:5d} {pb[j]:5d}  d={dist[near][j]:2d}  gap={seq_gap[j]}")
    STATS["near_duplicates"] = {
        "threshold": cfg.PHASH_THRESHOLD, "n_pairs": int(len(dist)),
        "mean_hamming": float(dist.mean()), "min_hamming": int(dist.min()),
        "n_near_dup": int(len(near)),
        "n_near_dup_adjacent": int((seq_gap <= 3).sum()) if len(near) else 0,
        "n_near_dup_far": int((seq_gap > 20).sum()) if len(near) else 0,
    }
else:
    print("no imagery cached -- skipped")

## 18. Split composition

The exact splitters from the baselines, reproduced verbatim so that the composition table in
Section III matches the folds that Section VI reports on. Nothing is trained; this only describes
what each fold contains.

Per fold and per mode: train/val tile counts, buffered-out tiles, and the **validation class
share**. The critical column is River — a fold whose validation block contains almost no River
cannot measure River IoU meaningfully, and any per-fold comparison that ignores this is comparing
different problems. This table is the evidence for that statement.

In [ ]:
def block_split(base_ids, fold, n_folds, buffer):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

def random_split(base_ids, fold, n_folds, seed):
    ids = list(base_ids); random.Random(seed).shuffle(ids)
    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    val = ids[edges[fold]:edges[fold + 1]]
    train = ids[:edges[fold]] + ids[edges[fold + 1]:]
    return sorted(train), sorted(val)

pos = {b: i for i, b in enumerate(BASE_IDS)}
rows = []
for mode in ("block", "random"):
    for k in range(cfg.N_FOLDS):
        tr, va = (block_split(BASE_IDS, k, cfg.N_FOLDS, cfg.BLOCK_BUFFER) if mode == "block"
                  else random_split(BASE_IDS, k, cfg.N_FOLDS, cfg.SEED))
        vi = [pos[b] for b in va]; ti = [pos[b] for b in tr]
        vs = CNT512[vi][:, :6].sum(0); vs = vs / vs.sum() * 100
        ts = CNT512[ti][:, :6].sum(0); ts = ts / ts.sum() * 100
        r = {"mode": mode, "fold": k, "train_tiles": len(tr), "val_tiles": len(va),
             "buffered": len(BASE_IDS) - len(tr) - len(va),
             "train_imgs": len(tr) * 3, "val_imgs": len(va) * 3,
             "val_id_lo": min(va), "val_id_hi": max(va),
             "val_tiles_with_river": int((CNT512[vi][:, 2] > 0).sum())}
        for c, nm in enumerate(cfg.CLASS_NAMES):
            r[f"val_{nm}_pct"] = vs[c]; r[f"train_{nm}_pct"] = ts[c]
        rows.append(r)
spl = pd.DataFrame(rows)
cols = ["mode", "fold", "train_tiles", "val_tiles", "buffered", "val_tiles_with_river"] + \
       [f"val_{n}_pct" for n in cfg.CLASS_NAMES]
print(spl[cols].to_string(index=False, float_format=lambda v: f"{v:8.2f}"))
print("\nval-set River share by fold (block):",
      [f"f{r.fold}: {r.val_River_pct:.3f}%" for r in spl[spl['mode']=='block'].itertuples()])
spl.to_csv(f"{cfg.OUT_DIR}/split_composition.csv", index=False)
STATS["splits"] = spl.to_dict("records")

## 19. Export-order structure

The block split rests on one claim: consecutive export ids are spatially close, so contiguous
blocks approximate spatial blocks. That claim is testable without any geo-coordinates.

If export order carries spatial structure, then class composition should be **autocorrelated along
the id axis** — neighbouring tiles should resemble each other more than random pairs do. Two
measures:

- lag-1 Pearson correlation of each class's per-tile share along the id sequence;
- mean absolute difference in the full 6-vector of shares between consecutive tiles, against the
  same quantity for randomly paired tiles. A ratio well below 1 means real ordering structure.

A strong result here is the honest justification for the block protocol; a null result is *also*
publishable and belongs in the limitations paragraph, since it would mean the block split is barely
different from a random one, which is consistent with the small leakage gap the baselines report.

In [ ]:
lag1 = {}
for c, nm in enumerate(cfg.CLASS_NAMES):
    x = SHARE[:-1, c]; y = SHARE[1:, c]
    lag1[nm] = float(np.corrcoef(x, y)[0, 1]) if x.std() > 0 and y.std() > 0 else float("nan")
print("lag-1 autocorrelation of per-tile class share along export order:")
for nm, v in lag1.items():
    print(f"  {nm:<14}{v:7.3f}")

adj = np.abs(SHARE[1:] - SHARE[:-1]).sum(1).mean()
rs = np.random.RandomState(cfg.SEED)
i1 = rs.randint(0, len(SHARE), 20000); i2 = rs.randint(0, len(SHARE), 20000)
rnd = np.abs(SHARE[i1] - SHARE[i2]).sum(1).mean()
print(f"\nmean L1 distance between class-share vectors:")
print(f"  consecutive tiles : {adj:.4f}")
print(f"  random tile pairs : {rnd:.4f}")
print(f"  ratio             : {adj/rnd:.3f}   "
      f"({'ordering carries structure' if adj/rnd < 0.9 else 'little ordering structure'})")

roll = pd.DataFrame(SHARE * 100, columns=list(cfg.CLASS_NAMES)).rolling(
    cfg.ROLL_WINDOW, center=True, min_periods=5).mean()
STATS["export_order"] = {"lag1_autocorr": lag1, "l1_consecutive": float(adj),
                         "l1_random": float(rnd), "ratio": float(adj / rnd)}

## 20. Figures for the paper

Six figures, written to `figs/` as 300-dpi PNG **and** vector PDF (IEEE templates prefer the PDF).
Panel titles are deliberately plain so they can be replaced by LaTeX captions.

1. `fig_samples` — image / mask / overlay triplets for three tiles, chosen as the most
   class-diverse in the dataset rather than at random, since a sample figure exists to show the
   annotation scheme.
2. `fig_class_distribution` — log-scale class share bar chart.
3. `fig_patch_area` — patch-area distribution per class, log axis.
4. `fig_order_structure` — rolling class share along export order.
5. `fig_cooccurrence` — tile-level Jaccard heatmap.
6. `fig_separability` — Jeffries-Matusita heatmap.

In [ ]:
def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{cfg.FIG_DIR}/{name}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("  wrote", name)

from matplotlib.patches import Patch
LEG = [Patch(facecolor=np.array(CLASS_TO_BGR[i][::-1]) / 255, label=cfg.CLASS_NAMES[i])
       for i in range(6)]

# 1 -- sample triplets
pick = np.argsort(-(PRESENT.sum(1) * 100 + (SHARE[:, 2] + SHARE[:, 1]) * 10))[:3]
fig, ax = plt.subplots(3, 3, figsize=(9, 9))
for r, t in enumerate(pick):
    rgb = np.ascontiguousarray(IMG512[t][..., ::-1])
    lab = np.ascontiguousarray(colorize(LAB512[t]))
    ax[r, 0].imshow(rgb); ax[r, 1].imshow(lab)
    ax[r, 2].imshow(cv2.addWeighted(rgb, 0.55, lab, 0.45, 0))
    for c, ttl in enumerate(["Image", "Annotation", "Overlay"]):
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        if r == 0: ax[r, c].set_title(ttl)
    ax[r, 0].set_ylabel(f"tile {BASE_IDS[t]}")
fig.legend(handles=LEG, loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.02))
save(fig, "fig_samples")

# 2 -- class distribution
fig, a = plt.subplots(figsize=(6, 3.2))
a.bar(range(6), p5[:6], color=[np.array(CLASS_TO_BGR[i][::-1]) / 255 for i in range(6)],
      edgecolor="k", linewidth=0.5)
for i, v in enumerate(p5[:6]): a.text(i, v * 1.06, f"{v:.2f}", ha="center", fontsize=8)
a.set_yscale("log"); a.set_ylabel("pixel share (%, log)")
a.set_xticks(range(6)); a.set_xticklabels(cfg.CLASS_NAMES, rotation=30, ha="right")
save(fig, "fig_class_distribution")

# 3 -- patch areas
if cfg.PATCH_STATS:
    fig, a = plt.subplots(figsize=(6, 3.2))
    data = [(mpa[PRESENT[:, c], c] * GSD_WORK ** 2) for c in range(6)]
    bp = a.boxplot(data, showfliers=False, patch_artist=True)
    a.set_xticks(range(1, 7)); a.set_xticklabels(cfg.CLASS_NAMES)
    for i, b in enumerate(bp["boxes"]): b.set_facecolor(np.array(CLASS_TO_BGR[i][::-1]) / 255)
    a.set_yscale("log"); a.set_ylabel("mean patch area per tile (m$^2$, log)")
    plt.setp(a.get_xticklabels(), rotation=30, ha="right")
    save(fig, "fig_patch_area")

# 4 -- export-order structure
fig, a = plt.subplots(figsize=(7, 3.2))
for c, nm in enumerate(cfg.CLASS_NAMES):
    a.plot(BASE_IDS, roll[nm], lw=1.2, color=np.array(CLASS_TO_BGR[c][::-1]) / 255, label=nm)
a.set_xlabel("base tile id (export order)"); a.set_ylabel(f"class share (%), rolling {cfg.ROLL_WINDOW}")
a.legend(ncol=3, fontsize=7, frameon=False)
save(fig, "fig_order_structure")

# 5, 6 -- heatmaps
for mat, nm, ttl, vmax in [(jac, "fig_cooccurrence", "Tile-level Jaccard co-occurrence", 1.0),
                           (JM, "fig_separability", "Jeffries-Matusita separability", 2.0)]:
    fig, a = plt.subplots(figsize=(5.2, 4.4))
    im = a.imshow(mat, cmap="viridis", vmin=0, vmax=vmax)
    a.set_xticks(range(6)); a.set_yticks(range(6))
    a.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); a.set_yticklabels(cfg.CLASS_NAMES)
    for i in range(6):
        for j in range(6):
            a.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", fontsize=7,
                   color="w" if mat[i, j] < vmax * 0.6 else "k")
    a.set_title(ttl); fig.colorbar(im, fraction=0.046)
    save(fig, nm)

## 21. Export — JSON, CSVs and LaTeX

Everything measured is written to `dataset_stats/`. The LaTeX file contains two ready-to-paste
table bodies (class distribution and dataset specification) using `booktabs`, so Section III can be
assembled without retyping a single number. The final block prints the sentences-worth of facts
that the section actually needs, in the order they are usually written.

In [ ]:
STATS["_meta"] = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"),
                  "notebook": "geonusaf-dataset-characterisation",
                  "cfg": {k: (list(v) if isinstance(v, tuple) else v)
                          for k, v in asdict(cfg).items()}}

def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
json.dump(STATS, open(f"{cfg.OUT_DIR}/dataset_stats.json", "w"), indent=2, default=_j)

spec = [("Study area", "Kathmandu Valley, Nepal"),
        ("Tile footprint", f"{cfg.TILE_SIDE_M:.0f} m $\\times$ {cfg.TILE_SIDE_M:.0f} m ({TILE_AREA_HA:.2f} ha)"),
        ("Native raster", f"{NAT_W} $\\times$ {NAT_H} px ({GSD_NATIVE*100:.1f} cm/px)"),
        ("Working raster", f"{cfg.IMG_SIZE} $\\times$ {cfg.IMG_SIZE} px ({GSD_WORK:.3f} m/px)"),
        ("Annotated tiles", f"{len(BASE_IDS)} ({ANNOT_KM2:.2f} km$^2$)"),
        ("Image--mask pairs", f"{len(records)} ({len(BASE_IDS)} $\\times$ 3 flip variants)"),
        ("Classes", f"{cfg.N_CLASSES} + ignore (255)"),
        ("Unlabelled share", f"{STATS['palette']['unmapped_pixel_share_pct']:.3f}\\%"),
        ("Imbalance ratio", f"{STATS['class_distribution']['imbalance_ratio']:.0f}:1")]
LATEX["dataset_spec"] = "\n".join(f"{k} & {v} \\\\" for k, v in spec)
with open(f"{cfg.OUT_DIR}/latex_tables.tex", "w") as fh:
    for k, v in LATEX.items():
        fh.write(f"% ---- {k} ----\n{v}\n\n")

print("written to", cfg.OUT_DIR)
for f in sorted(os.listdir(cfg.OUT_DIR)): print("  ", f)
print("figures:")
for f in sorted(os.listdir(cfg.FIG_DIR)): print("  ", f)

print("\n" + "=" * 78)
print("SECTION III FACT SHEET")
print("=" * 78)
g_, i_, c_ = STATS["geometry"], STATS["inventory"], STATS["class_distribution"]
print(f"Kathmandu Valley | {i_['n_base_tiles']} annotated tiles of "
      f"{cfg.TILE_SIDE_M:.0f}x{cfg.TILE_SIDE_M:.0f} m = {g_['annotated_area_km2']:.2f} km^2 "
      f"(swept extent {g_['swept_extent_km2']:.2f} km^2, "
      f"{i_['n_dropped_ids']} mono-class ids dropped)")
print(f"Rasters {NAT_W}x{NAT_H} px, GSD {g_['gsd_native_m']*100:.2f} cm/px native, "
      f"{g_['gsd_working_m']:.3f} m/px at the {cfg.IMG_SIZE} working resolution")
print(f"{i_['n_pairs']} image-mask pairs = {i_['n_base_tiles']} scenes x 3 flip variants; "
      f"{i_['size_total_gb']:.2f} GB on disk")
print(f"{cfg.N_CLASSES} classes + ignore; unlabelled {STATS['palette']['unmapped_pixel_share_pct']:.3f}% "
      f"over {STATS['palette']['total_label_pixels']/1e9:.2f}e9 label pixels; "
      f"{STATS['palette']['n_distinct_colours']} distinct mask colours")
print("class shares @512 : " + " | ".join(f"{n} {p5[i]:.2f}%" for i, n in enumerate(cfg.CLASS_NAMES)))
print(f"imbalance ratio {c_['imbalance_ratio']:.0f}:1 | effective classes "
      f"{c_['effective_classes']:.2f}/6 | mean classes per tile {STATS['per_tile']['classes_per_tile_mean']:.2f}")
if "thin_structures" in STATS:
    for nm, d in STATS["thin_structures"].items():
        print(f"{nm}: median width {d['median_width_m']:.2f} m ({d['median_width_px']:.1f} px @512), "
              f"{d['total_length_km']:.0f} km total, {d['length_density_km_per_km2']:.2f} km/km^2")
if "spectral" in STATS:
    v, a, b = STATS["spectral"]["least_separable"][0]
    print(f"least separable pair: {a} vs {b}, JM = {v:.3f}")
print(f"export-order structure: consecutive/random L1 ratio "
      f"{STATS['export_order']['ratio']:.3f}")
print("=" * 78)

## 22. Run checklist

1. Attach `sugam24/satellite-datasets`. Internet and GPU are **not** needed.
2. Run top to bottom. Section 4 must print **channel-order check PASSED** before section 6 starts.
3. Section 6 is the only slow cell (12-18 min first run). It writes `dsstats_cache.npz`; later runs
   reload it in seconds. If the printed tile order assertion fails, delete the npz and re-run.
4. Section 3 must print a **single** raster size. More than one means the GSD is not constant and
   every metre-denominated number below is invalid.
5. Section 16 must report the variants as exact mirrors. If it does not, stop using 408 as the
   reporting basis and re-derive the statistics over all 1224 files.
6. `Save Version` -> download `dataset_stats/` from the output pane. That directory is the input to
   writing Section III: `dataset_stats.json` for prose numbers, the CSVs for the tables,
   `latex_tables.tex` for direct paste, `figs/*.pdf` for the figures.

### What this notebook cannot know
Provenance is not recoverable from the files. The imagery source and acquisition date, the CRS and
bounding box of the Atlas coverage layer, the tile overlap setting, the annotation workflow and its
quality control, and the class definitions themselves all have to be supplied by hand. Those are
exactly the parts of Section III that a reviewer will read most carefully, so they should be
written from records rather than from memory.